# Tutorial 57: Bulk Result Read, Output Factors, and Time Curve Data

This example demonstrates how to read calculation results quickly with generate_element_results_dataframe(bulk=True): instead of one Toolkit API call per value, the values are read directly from the result files (.MX1/.MX2/.MXS). This pays off for every request with more than a few elements or timestamps, and makes reading very large result files possible. It also allows for the specification of an output factor file to read and therefore the reading of time curve data directly.

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-11-25 16:25:24,544] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:25:24,544] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:25:24,569] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:25:24,687] INFO in sir3stoolkit.core.wrapper: [Initialization] Sir3S_Toolkit.dll file version 90.15.11.0 (matches this sir3stoolkit version).


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-11-25 16:25:26,123] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


# Open Model

The tutorial uses *Network 3*, a fictional district heating network shared by the mantle tutorials: a heat plant (in its own container "Heizwerk"), about 180 consumers, a ring road and two longitudinal sections, calculated over one day. It is built from scratch with the SIR 3S Toolkit by [Network3.py](https://github.com/3SConsult/sir3stoolkit/blob/main/docs/source/networks/Network3.py); `Network3.db3` is included in the tutorial's ZIP archive.

Its elements have unique ID references (IDREFERENZ), e.g. `VL-HS-01` for the first supply pipe of the main street. They are used below to look up tks with [GetTkFromIDReference()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetTkFromIDReference) - the tks themselves differ between builds of the model.

In [8]:
s3s.OpenModel(dbName=r"Network3.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-11-25 16:25:31,876] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Calculate Model

In [9]:
s3s.ExecCalculation(True) # To ensure result data

[2026-11-25 16:25:36,855] INFO in sir3stoolkit.core.wrapper: Model Calculation is complete


# Bulk Read

## Basics

The regular [generate_element_results_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_results_dataframe) function utilizes the Toolkit's most basic functions like [GetResultfortimestamp()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetResultfortimestamp) in a Python loop to retrieve the result values for the dataframe. This can be quite slow.

In [10]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe, timestamps=[0, -1]) # slow: one Toolkit API call per value

[2026-11-25 16:25:36,871] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:25:36,937] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 2 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-16 00:00:00.000 +01:00']


[2026-11-25 16:25:36,942] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:25:36,955] INFO in sir3stoolkit.mantle.dataframes: [results] No properties given → using ALL result properties for ObjectTypes.Pipe.


[2026-11-25 16:25:36,955] INFO in sir3stoolkit.mantle.dataframes: [results] Using 82 result properties.


[2026-11-25 16:25:36,955] INFO in sir3stoolkit.mantle.dataframes: [results] 2 timestamp(s), 572 element(s), 82 properties resolved (0.1 s).


... (7 outputs left out)

[2026-11-25 16:26:31,472] INFO in sir3stoolkit.mantle.dataframes: [results] 30316 fully NaN columns dropped.


[2026-11-25 16:26:31,472] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.7 s).


[2026-11-25 16:26:31,545] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:26:31,554] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (2, 16588) (54.7 s in total)


In [11]:
df_results.head(3)

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                                A   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                                0.0           
2026-01-16 00:00:00.000 +01:00                                                0.0           

tk                                                                           \
name                                                                          
end_nodes                                                                     
property                            DTTR    DWVERL DWVERLABS IAKTIV IRTRENN   
timestamp                                                                     
2026-01-15 00:00:00.000 +01:00  0.053661  26.19290  2.619290    0.0     0.0   
2026-01-16 00:00:00.000 +01:00  0.053662  26.17971  2.617971    0.0     0.0   

tk                                                                      \
name                                                                     
end_nodes                                                                
property                              JV                MVEC    PDAMPF   
timestamp                                                                
2026-01-15 00:00:00.000 +01:00  0.084540  27.24994\t27.24994  0.577745   
2026-01-16 00:00:00.000 +01:00  0.084536  27.24931\t27.24931  0.577745   

tk                                        ...  \
name                                      ...   
end_nodes                                 ...   
property                             PHR  ...   
timestamp                                 ...   
2026-01-15 00:00:00.000 +01:00  0.008454  ...   
2026-01-16 00:00:00.000 +01:00  0.008454  ...   

tk                                                                    4622768628845952753  \
name                                                            Rohr R-H-RATHAUS R-KAS-04   
end_nodes                      ('5761893596493624260', '5007908863305915354', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                      0\t0.05199681           
2026-01-16 00:00:00.000 +01:00                                      0\t0.05198273           

tk                                                                          \
name                                                                         
end_nodes                                                                    
property                                TVEC       VAV        VI        VK   
timestamp                                                                    
2026-01-15 00:00:00.000 +01:00  60\t59.81641  0.160266  0.160274  0.160259   
2026-01-16 00:00:00.000 +01:00  60\t59.81638  0.160310  0.160317  0.160302   

tk                                                                   
name                                                                 
end_nodes                                                            
property                       VOLDA  WVL ZAUS ZEIN            ZVEC  
timestamp                                                            
2026-01-15 00:00:00.000 +01:00   0.0  0.0  0.0  0.0  210.14\t210.97  
2026-01-16 00:00:00.000 +01:00   0.0  0.0  0.0  0.0  210.14\t210.97  

[2 rows x 16588 columns]

Instead we can use the parameter bulk=True to read the mx result files directly. This is much faster and especially relevant for larger models.

In [12]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe, bulk=True, timestamps=[0, -1]) # fast: reads the result files directly

[2026-11-25 16:26:31,622] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:26:31,650] INFO in sir3stoolkit.mantle.mx: [mx] M-1-0-1.2.MXS: 189 channels, 16 records of 162,900 bytes (0.00 GiB).


[2026-11-25 16:26:31,661] INFO in sir3stoolkit.mantle.dataframes: [bulk results] Result file: M-1-0-1.2.MX1 (output factor 2 = DTFAK_GROSS of the most recent calculation, default), 13 timestamps.


[2026-11-25 16:26:31,663] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:26:31,665] INFO in sir3stoolkit.mantle.dataframes: [results] Using 29 result properties from the result file.


[2026-11-25 16:26:31,666] INFO in sir3stoolkit.mantle.dataframes: [results] 2 timestamp(s), 572 element(s), 29 properties resolved (0.0 s).


... (2 outputs left out)

[2026-11-25 16:26:31,738] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:26:31,740] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:26:31,803] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:26:31,803] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (2, 16588) (0.2 s in total)


The direct byte-read of the mx files is much faster. In addition only result properties that exist in the calculation results will be attempted to be read.

Still keep in mind that if you have a very large model ~ two digits of GiBs in .MXS, pandas will need to hold a dataframe of similar size in RAM if the entire file is requested. For oversized requests the function raises a MemoryError before reading anything, instead of failing during execution. You should always try to keep dimension tks X timestamps X properties to the data you actually need. 

In [13]:
df_results.head(3)

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                                A   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                                0.0           
2026-01-16 00:00:00.000 +01:00                                                0.0           

tk                                                                            \
name                                                                           
end_nodes                                                                      
property                            DTTR     DWVERL DWVERLABS IAKTIV IRTRENN   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  0.053661  26.192898  2.619290    0.0     0.0   
2026-01-16 00:00:00.000 +01:00  0.053662  26.179714  2.617971    0.0     0.0   

tk                                                                      \
name                                                                     
end_nodes                                                                
property                              JV                MVEC    PDAMPF   
timestamp                                                                
2026-01-15 00:00:00.000 +01:00  0.084540  27.24994\t27.24994  0.577745   
2026-01-16 00:00:00.000 +01:00  0.084536  27.24931\t27.24931  0.577745   

tk                                        ...  \
name                                      ...   
end_nodes                                 ...   
property                             PHR  ...   
timestamp                                 ...   
2026-01-15 00:00:00.000 +01:00  0.008454  ...   
2026-01-16 00:00:00.000 +01:00  0.008454  ...   

tk                                                                    4622768628845952753  \
name                                                            Rohr R-H-RATHAUS R-KAS-04   
end_nodes                      ('5761893596493624260', '5007908863305915354', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                      0\t0.05199681           
2026-01-16 00:00:00.000 +01:00                                      0\t0.05198273           

tk                                                                          \
name                                                                         
end_nodes                                                                    
property                                TVEC       VAV        VI        VK   
timestamp                                                                    
2026-01-15 00:00:00.000 +01:00  60\t59.81641  0.160266  0.160274  0.160259   
2026-01-16 00:00:00.000 +01:00  60\t59.81638  0.160310  0.160317  0.160302   

tk                                                                   
name                                                                 
end_nodes                                                            
property                       VOLDA  WVL ZAUS ZEIN            ZVEC  
timestamp                                                            
2026-01-15 00:00:00.000 +01:00   0.0  0.0  0.0  0.0  210.14\t210.97  
2026-01-16 00:00:00.000 +01:00   0.0  0.0  0.0  0.0  210.14\t210.97  

[2 rows x 16588 columns]

## Interior Points Formatting

Interior points make up a large share of the result data and of the read time, so you should only request them if they are actually required.

### str

By default interior point result properties with bulk=True are formatted as tab-separated strings to be compatible with the format from the bulk=False read.

In [14]:
tk_of_interest = s3s.GetTkFromIDReference("VL-HS-01", s3s.ObjectTypes.Pipe) # first supply pipe of the main street

In [15]:
timestamp_of_interest = "2026-01-15 00:00:00.000 +01:00"

In [16]:
vec = df_results.xs((tk_of_interest, "PVEC"), axis=1, level=["tk", "property"]).loc[timestamp_of_interest]

In [17]:
vec.iloc[0]

'8.636661\t8.64151'

#### merge with model data

In [18]:
df_model_data = s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Pipe, properties=["Di"])

[2026-11-25 16:26:31,875] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:26:31,879] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:26:31,879] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 1 model_data properties.


[2026-11-25 16:26:31,882] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Di']...


[2026-11-25 16:26:31,911] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (572, 2)


In [19]:
df_results = s3s.add_interior_points_to_start_end_sequence(df_results)

In [20]:
df_results, timestamp_to_tuple_index = s3s.convert_rows_to_single_tuple_row(df_results)

[2026-11-25 16:26:35,335] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Collapsing dataframe rows into tuple row...


[2026-11-25 16:26:36,097] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Done. Shape: (1, 26884)


In [21]:
df = s3s.merge_model_data_and_results(df_model_data, df_results)

In [22]:
df.head()

,tk,Di,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,...,VAV,VI,VK,VOLDA,WVL,ZAUS,ZEIN,ZVEC_end,ZVEC_sequence,ZVEC_start
0,4612274316321154308,30.4,"(0.0, 0.0)","(0.017941685, 0.017938588)","(13.764609, 13.765386)","(0.20646915, 0.20648079)","(0.0, 0.0)","(0.0, 0.0)","(0.6026315, 0.602839)","(0.163439, 0.1634675)",...,"(0.23223385, 0.23227394)","(0.23225638, 0.23229717)","(0.23220964, 0.23225045)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(208.7, 208.7)","((208.31, 208.7), (208.31, 208.7))","(208.31, 208.31)"
1,4614940901611765838,30.4,"(0.0, 0.0)","(0.017313076, 0.017308816)","(13.841067, 13.84068)","(0.20761602, 0.2076102)","(0.0, 0.0)","(0.0, 0.0)","(0.6470199, 0.64733833)","(0.1693296, 0.1693717)",...,"(0.24066588, 0.24072511)","(0.24068691, 0.24074681)","(0.24064346, 0.2407033)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(197.97, 197.97)","((198.04, 197.97), (198.04, 197.97))","(198.04, 198.04)"
2,4618102496224966047,30.4,"(0.0, 0.0)","(0.039666574, 0.039651792)","(13.821736, 13.821945)","(0.20732605, 0.20732918)","(0.0, 0.0)","(0.0, 0.0)","(0.12326645, 0.12335833)","(0.07391112, 0.07393897)",...,"(0.105042264, 0.10508142)","(0.10506366, 0.10510327)","(0.105019934, 0.105059534)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(200.24, 200.24)","((200.32, 200.24), (200.32, 200.24))","(200.32, 200.32)"
3,4618494560130677338,82.5,"(0.0, 0.0)","(0.022687588, 0.022679714)","(11.63308, 11.637901)","(0.581654, 0.58189505)","(0.0, 0.0)","(0.0, 0.0)","(0.5084495, 0.5087894)","(3.219939, 3.221066)",...,"(0.61218005, 0.6123926)","(0.6121851, 0.6123993)","(0.6121717, 0.61238587)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(199.02, 199.02)","((199.72, 199.02), (199.72, 199.02))","(199.72, 199.72)"
4,4619202338427165336,54.5,"(0.0, 0.0)","(0.038229566, 0.038274378)","(14.316172, 14.317654)","(0.7158086, 0.7158827)","(0.0, 0.0)","(0.0, 0.0)","(0.3026995, 0.30202326)","(0.821668, 0.8207069)",...,"(0.3633023, 0.36287695)","(0.36332485, 0.36290008)","(0.363278, 0.36285344)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(201.1, 201.1)","((199.92, 201.1), (199.92, 201.1))","(199.92, 199.92)"


### multiindex

But for even faster result retrieval use vec_format = "multiindex". Every interior point has its own sub column. This scales for larger models and can have quite a large time saving effect. 

Unless you still rely on some older toolkit code using the str formatting to merge the results with model data, multiindex formatting is superior.

In [23]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe, bulk=True, timestamps=[0, -1], vec_format="multiindex")

[2026-11-25 16:26:36,203] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:26:36,207] INFO in sir3stoolkit.mantle.dataframes: [bulk results] Result file: M-1-0-1.2.MX1 (output factor 2 = DTFAK_GROSS of the most recent calculation, default), 13 timestamps.


[2026-11-25 16:26:36,210] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:26:36,210] INFO in sir3stoolkit.mantle.dataframes: [results] Using 29 result properties from the result file.


[2026-11-25 16:26:36,210] INFO in sir3stoolkit.mantle.dataframes: [results] 2 timestamp(s), 572 element(s), 29 properties resolved (0.0 s).


[2026-11-25 16:26:36,210] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=True)


[2026-11-25 16:26:36,222] INFO in sir3stoolkit.mantle.dataframes: [bulk results] 29 properties read from M-1-0-1.2.MX1 (0.0 s).


[2026-11-25 16:26:36,222] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:26:36,225] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:26:36,281] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:26:36,292] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (2, 21880) (0.1 s in total)


In [24]:
df_results.head()

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                                A   
interior points                                                                        -1   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                                0.0           
2026-01-16 00:00:00.000 +01:00                                                0.0           

tk                                                                            \
name                                                                           
end_nodes                                                                      
property                            DTTR     DWVERL DWVERLABS IAKTIV IRTRENN   
interior points                       -1         -1        -1     -1      -1   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  0.053661  26.192898  2.619290    0.0     0.0   
2026-01-16 00:00:00.000 +01:00  0.053662  26.179714  2.617971    0.0     0.0   

tk                                                                        ...  \
name                                                                      ...   
end_nodes                                                                 ...   
property                              JV       MVEC               PDAMPF  ...   
interior points                       -1          0          1        -1  ...   
timestamp                                                                 ...   
2026-01-15 00:00:00.000 +01:00  0.084540  27.249939  27.249939  0.577745  ...   
2026-01-16 00:00:00.000 +01:00  0.084536  27.249308  27.249308  0.577745  ...   

tk                                                                    4622768628845952753  \
name                                                            Rohr R-H-RATHAUS R-KAS-04   
end_nodes                      ('5761893596493624260', '5007908863305915354', '-1', '-1')   
property                                                                             TVEC   
interior points                                                                         1   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                          59.816406           
2026-01-16 00:00:00.000 +01:00                                          59.816376           

tk                                                                            \
name                                                                           
end_nodes                                                                      
property                             VAV        VI        VK VOLDA  WVL ZAUS   
interior points                       -1        -1        -1    -1   -1   -1   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  0.160266  0.160274  0.160259   0.0  0.0  0.0   
2026-01-16 00:00:00.000 +01:00  0.160310  0.160317  0.160302   0.0  0.0  0.0   

tk                                                           
name                                                         
end_nodes                                                    
property                       ZEIN        ZVEC              
interior points                  -1           0           1  
timestamp                                                    
2026-01-15 00:00:00.000 +01:00  0.0  210.139999  210.970001  
2026-01-16 00:00:00.000 +01:00  0.0  210.139999  210.970001  

[2 rows x 21880 columns]

Note that the access to the interior points therefore also changes.

In [25]:
vec = df_results.xs((tk_of_interest, "PVEC"), axis=1, level=["tk", "property"]).loc[timestamp_of_interest].to_list()

In [26]:
type(vec)

list

In [27]:
vec

[8.6366605758667, 8.641510009765625]

The data access can also be facilitated if you drop index levels "name" and "end_nodes". 

In [28]:
df_results.columns = df_results.columns.droplevel(["name", "end_nodes"])

Using other methods of removing the levels might take long time and memory.

In [29]:
df_results.head()

tk                             5448887806005001177                       \
property                                         A      DTTR     DWVERL   
interior points                                 -1        -1         -1   
timestamp                                                                 
2026-01-15 00:00:00.000 +01:00                 0.0  0.053661  26.192898   
2026-01-16 00:00:00.000 +01:00                 0.0  0.053662  26.179714   

tk                                                                            \
property                       DWVERLABS IAKTIV IRTRENN        JV       MVEC   
interior points                       -1     -1      -1        -1          0   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  2.619290    0.0     0.0  0.084540  27.249939   
2026-01-16 00:00:00.000 +01:00  2.617971    0.0     0.0  0.084536  27.249308   

tk                                                   ... 4622768628845952753  \
property                                     PDAMPF  ...                TVEC   
interior points                         1        -1  ...                   1   
timestamp                                            ...                       
2026-01-15 00:00:00.000 +01:00  27.249939  0.577745  ...           59.816406   
2026-01-16 00:00:00.000 +01:00  27.249308  0.577745  ...           59.816376   

tk                                                                            \
property                             VAV        VI        VK VOLDA  WVL ZAUS   
interior points                       -1        -1        -1    -1   -1   -1   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  0.160266  0.160274  0.160259   0.0  0.0  0.0   
2026-01-16 00:00:00.000 +01:00  0.160310  0.160317  0.160302   0.0  0.0  0.0   

tk                                                           
property                       ZEIN        ZVEC              
interior points                  -1           0           1  
timestamp                                                    
2026-01-15 00:00:00.000 +01:00  0.0  210.139999  210.970001  
2026-01-16 00:00:00.000 +01:00  0.0  210.139999  210.970001  

[2 rows x 21880 columns]

In [30]:
vec = df_results.loc[timestamp_of_interest, (tk_of_interest, "PVEC")].to_list()

In [31]:
vec

[8.6366605758667, 8.641510009765625]

#### merge with model data

In [32]:
df = s3s.merge_model_data_and_results(df_model_data, df_results)

In [33]:
df.head()

,tk,Di,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,...,VAV,VI,VK,VOLDA,WVL,ZAUS,ZEIN,ZVEC_end,ZVEC_sequence,ZVEC_start
0,4612274316321154308,30.4,"(0.0, 0.0)","(0.017941685, 0.017938588)","(13.764609, 13.765386)","(0.20646915, 0.20648079)","(0.0, 0.0)","(0.0, 0.0)","(0.6026315, 0.602839)","(0.16343898, 0.16346753)",...,"(0.23223385, 0.23227394)","(0.23225638, 0.23229717)","(0.23220964, 0.23225045)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(208.7, 208.7)","((208.31, 208.7), (208.31, 208.7))","(208.31, 208.31)"
1,4614940901611765838,30.4,"(0.0, 0.0)","(0.017313076, 0.017308816)","(13.841067, 13.84068)","(0.20761602, 0.2076102)","(0.0, 0.0)","(0.0, 0.0)","(0.6470199, 0.64733833)","(0.16932957, 0.16937166)",...,"(0.24066588, 0.24072511)","(0.24068691, 0.24074681)","(0.24064346, 0.2407033)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(197.97, 197.97)","((198.04, 197.97), (198.04, 197.97))","(198.04, 198.04)"
2,4618102496224966047,30.4,"(0.0, 0.0)","(0.039666574, 0.039651792)","(13.821736, 13.821945)","(0.20732605, 0.20732918)","(0.0, 0.0)","(0.0, 0.0)","(0.12326645, 0.12335833)","(0.07391112, 0.07393897)",...,"(0.105042264, 0.10508142)","(0.10506366, 0.10510327)","(0.105019934, 0.105059534)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(200.24, 200.24)","((200.32, 200.24), (200.32, 200.24))","(200.32, 200.32)"
3,4618494560130677338,82.5,"(0.0, 0.0)","(0.022687588, 0.022679714)","(11.63308, 11.637901)","(0.581654, 0.58189505)","(0.0, 0.0)","(0.0, 0.0)","(0.5084495, 0.5087894)","(3.2199388, 3.2210655)",...,"(0.61218005, 0.6123926)","(0.6121851, 0.6123993)","(0.6121717, 0.61238587)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(199.02, 199.02)","((199.72, 199.02), (199.72, 199.02))","(199.72, 199.72)"
4,4619202338427165336,54.5,"(0.0, 0.0)","(0.038229566, 0.038274378)","(14.316172, 14.317654)","(0.7158086, 0.7158827)","(0.0, 0.0)","(0.0, 0.0)","(0.3026995, 0.30202326)","(0.821668, 0.8207069)",...,"(0.3633023, 0.36287695)","(0.36332485, 0.36290008)","(0.363278, 0.36285344)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(201.1, 201.1)","((199.92, 201.1), (199.92, 201.1))","(199.92, 199.92)"


## Without Names and End Nodes

Looking up names and end nodes costs two API calls per element - on large models that can take longer than the bulk read itself. If you don't need them, set include_name_and_end_nodes=False: the columns are then just tk, property and interior points.

In [34]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe, bulk=True, timestamps=[0, -1], vec_format="multiindex", include_name_and_end_nodes=False)

[2026-11-25 16:26:36,598] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:26:36,601] INFO in sir3stoolkit.mantle.dataframes: [bulk results] Result file: M-1-0-1.2.MX1 (output factor 2 = DTFAK_GROSS of the most recent calculation, default), 13 timestamps.


[2026-11-25 16:26:36,603] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:26:36,603] INFO in sir3stoolkit.mantle.dataframes: [results] Using 29 result properties from the result file.


[2026-11-25 16:26:36,608] INFO in sir3stoolkit.mantle.dataframes: [results] 2 timestamp(s), 572 element(s), 29 properties resolved (0.0 s).


[2026-11-25 16:26:36,609] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=True)


[2026-11-25 16:26:36,612] INFO in sir3stoolkit.mantle.dataframes: [bulk results] 29 properties read from M-1-0-1.2.MX1 (0.0 s).


[2026-11-25 16:26:36,612] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:26:36,619] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:26:36,622] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (2, 21880) (0.0 s in total)


In [35]:
df_results.head()

tk                             5448887806005001177                       \
property                                         A      DTTR     DWVERL   
interior points                                 -1        -1         -1   
timestamp                                                                 
2026-01-15 00:00:00.000 +01:00                 0.0  0.053661  26.192898   
2026-01-16 00:00:00.000 +01:00                 0.0  0.053662  26.179714   

tk                                                                            \
property                       DWVERLABS IAKTIV IRTRENN        JV       MVEC   
interior points                       -1     -1      -1        -1          0   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  2.619290    0.0     0.0  0.084540  27.249939   
2026-01-16 00:00:00.000 +01:00  2.617971    0.0     0.0  0.084536  27.249308   

tk                                                   ... 4622768628845952753  \
property                                     PDAMPF  ...                TVEC   
interior points                         1        -1  ...                   1   
timestamp                                            ...                       
2026-01-15 00:00:00.000 +01:00  27.249939  0.577745  ...           59.816406   
2026-01-16 00:00:00.000 +01:00  27.249308  0.577745  ...           59.816376   

tk                                                                            \
property                             VAV        VI        VK VOLDA  WVL ZAUS   
interior points                       -1        -1        -1    -1   -1   -1   
timestamp                                                                      
2026-01-15 00:00:00.000 +01:00  0.160266  0.160274  0.160259   0.0  0.0  0.0   
2026-01-16 00:00:00.000 +01:00  0.160310  0.160317  0.160302   0.0  0.0  0.0   

tk                                                           
property                       ZEIN        ZVEC              
interior points                  -1           0           1  
timestamp                                                    
2026-01-15 00:00:00.000 +01:00  0.0  210.139999  210.970001  
2026-01-16 00:00:00.000 +01:00  0.0  210.139999  210.970001  

[2 rows x 21880 columns]

Merging still works with the same function.

In [36]:
df = s3s.merge_model_data_and_results(df_model_data, df_results)

In [37]:
df.head()

,tk,Di,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,...,VAV,VI,VK,VOLDA,WVL,ZAUS,ZEIN,ZVEC_end,ZVEC_sequence,ZVEC_start
0,4612274316321154308,30.4,"(0.0, 0.0)","(0.017941685, 0.017938588)","(13.764609, 13.765386)","(0.20646915, 0.20648079)","(0.0, 0.0)","(0.0, 0.0)","(0.6026315, 0.602839)","(0.16343898, 0.16346753)",...,"(0.23223385, 0.23227394)","(0.23225638, 0.23229717)","(0.23220964, 0.23225045)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(208.7, 208.7)","((208.31, 208.7), (208.31, 208.7))","(208.31, 208.31)"
1,4614940901611765838,30.4,"(0.0, 0.0)","(0.017313076, 0.017308816)","(13.841067, 13.84068)","(0.20761602, 0.2076102)","(0.0, 0.0)","(0.0, 0.0)","(0.6470199, 0.64733833)","(0.16932957, 0.16937166)",...,"(0.24066588, 0.24072511)","(0.24068691, 0.24074681)","(0.24064346, 0.2407033)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(197.97, 197.97)","((198.04, 197.97), (198.04, 197.97))","(198.04, 198.04)"
2,4618102496224966047,30.4,"(0.0, 0.0)","(0.039666574, 0.039651792)","(13.821736, 13.821945)","(0.20732605, 0.20732918)","(0.0, 0.0)","(0.0, 0.0)","(0.12326645, 0.12335833)","(0.07391112, 0.07393897)",...,"(0.105042264, 0.10508142)","(0.10506366, 0.10510327)","(0.105019934, 0.105059534)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(200.24, 200.24)","((200.32, 200.24), (200.32, 200.24))","(200.32, 200.32)"
3,4618494560130677338,82.5,"(0.0, 0.0)","(0.022687588, 0.022679714)","(11.63308, 11.637901)","(0.581654, 0.58189505)","(0.0, 0.0)","(0.0, 0.0)","(0.5084495, 0.5087894)","(3.2199388, 3.2210655)",...,"(0.61218005, 0.6123926)","(0.6121851, 0.6123993)","(0.6121717, 0.61238587)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(199.02, 199.02)","((199.72, 199.02), (199.72, 199.02))","(199.72, 199.72)"
4,4619202338427165336,54.5,"(0.0, 0.0)","(0.038229566, 0.038274378)","(14.316172, 14.317654)","(0.7158086, 0.7158827)","(0.0, 0.0)","(0.0, 0.0)","(0.3026995, 0.30202326)","(0.821668, 0.8207069)",...,"(0.3633023, 0.36287695)","(0.36332485, 0.36290008)","(0.363278, 0.36285344)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(201.1, 201.1)","((199.92, 201.1), (199.92, 201.1))","(199.92, 199.92)"


# Output Factors

In SIR 3S calculation settings the output factors large and small are determined. Unless they are identical two result files with output factor as postfix are created. Network 3 uses small = 1 (the time curves, every hour) and large = 2 (all elements, every 2 hours).

## Large

We can use the function [get_result_file_properties()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.get_result_file_properties) to check what properties are stored for a certain output factor.

In [38]:
s3s.get_result_file_properties(output_factor=2).head()

[2026-11-25 16:26:36,822] INFO in sir3stoolkit.mantle.dataframes: [results] M-1-0-1.2.MX1 (output factor 2 = DTFAK_GROSS of the most recent calculation, output_factor parameter): 138 result properties of 10 element type(s), 13 timestamps.


,element_type,objtype,property,unit,description,stored_for,result_file,n_timestamps
0,DistrictHeatingFeeder,FWES,DPH,[bar],Reibungsdruckverlust/-druckverlusthöhe,all elements,M-1-0-1.2.MX1,13
1,DistrictHeatingFeeder,FWES,IAKTIV,[],Netztrennstatus,all elements,M-1-0-1.2.MX1,13
2,DistrictHeatingFeeder,FWES,IHYTYP,[],Hydraulische Fahrweise,all elements,M-1-0-1.2.MX1,13
3,DistrictHeatingFeeder,FWES,ITHTYP,[],Thermische Fahrweise,all elements,M-1-0-1.2.MX1,13
4,DistrictHeatingFeeder,FWES,QM,[t/h],Durchfluss,all elements,M-1-0-1.2.MX1,13


By default generate_element_results_dataframe(bulk=True) reads the largest output factor file, or a file without an output factor if it existed.

In [39]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe, properties=["PVEC", "DTTR"], bulk=True, vec_format="multiindex")

[2026-11-25 16:26:36,851] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:26:36,854] INFO in sir3stoolkit.mantle.dataframes: [bulk results] Result file: M-1-0-1.2.MX1 (output factor 2 = DTFAK_GROSS of the most recent calculation, default), 13 timestamps.


[2026-11-25 16:26:36,854] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:26:36,860] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties from the result file.


[2026-11-25 16:26:36,860] INFO in sir3stoolkit.mantle.dataframes: [results] 13 timestamp(s), 572 element(s), 2 properties resolved (0.0 s).


[2026-11-25 16:26:36,860] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=True)


[2026-11-25 16:26:36,866] INFO in sir3stoolkit.mantle.dataframes: [bulk results] 2 properties read from M-1-0-1.2.MX1 (0.0 s).


[2026-11-25 16:26:36,866] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:26:36,866] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:26:36,924] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:26:36,931] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (13, 1732) (0.1 s in total)


In [40]:
df_results.index

Index(['2026-01-15 00:00:00.000 +01:00', '2026-01-15 02:00:00.000 +01:00',
       '2026-01-15 04:00:00.000 +01:00', '2026-01-15 06:00:00.000 +01:00',
       '2026-01-15 08:00:00.000 +01:00', '2026-01-15 10:00:00.000 +01:00',
       '2026-01-15 12:00:00.000 +01:00', '2026-01-15 14:00:00.000 +01:00',
       '2026-01-15 16:00:00.000 +01:00', '2026-01-15 18:00:00.000 +01:00',
       '2026-01-15 20:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00',
       '2026-01-16 00:00:00.000 +01:00'],
      dtype='object', name='timestamp')

## Small

Result properties in the file for the small output factor are determined by the time curves set in SIR 3S.

In [41]:
s3s.get_result_file_properties(output_factor=1)

[2026-11-25 16:26:36,969] INFO in sir3stoolkit.mantle.mx: [mx] M-1-0-1.1.MXS: 55 channels, 28 records of 324 bytes (0.00 GiB).


[2026-11-25 16:26:36,979] INFO in sir3stoolkit.mantle.dataframes: [results] M-1-0-1.1.MX1 (output factor 1 = DTFAK_KLEIN of the most recent calculation, output_factor parameter): 7 result properties of 4 element type(s), 25 timestamps.


,element_type,objtype,property,unit,description,stored_for,result_file,n_timestamps
0,DistrictHeatingFeeder,FWES,W,[kW],Wärmeleistung,single elements,M-1-0-1.1.MX1,25
1,Node,KNOT,PH,[bar],Druck,single elements,M-1-0-1.1.MX1,25
2,Node,KNOT,T,[°C],Temperatur,single elements,M-1-0-1.1.MX1,25
3,Pump,PUMP,QM,[t/h],Durchfluss,single elements,M-1-0-1.1.MX1,25
4,WBLZ_ThermalBalance,WBLZ,WES,[MW],Einspeiseleistung,single elements,M-1-0-1.1.MX1,25
5,WBLZ_ThermalBalance,WBLZ,WVB,[MW],Verbrauchsleistung Istwert,single elements,M-1-0-1.1.MX1,25
6,WBLZ_ThermalBalance,WBLZ,WVERL,[MW],Verlustleistung Rohrnetz,single elements,M-1-0-1.1.MX1,25


Then we can force the read of the output factor file using the param output_factor.

In [42]:
df_results = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.WBLZ_ThermalBalance, bulk=True, vec_format="multiindex", output_factor=1, properties=["WVB", "WVERL"])

[2026-11-25 16:26:36,992] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.WBLZ_ThermalBalance


[2026-11-25 16:26:36,995] INFO in sir3stoolkit.mantle.dataframes: [bulk results] Result file: M-1-0-1.1.MX1 (output factor 1 = DTFAK_KLEIN of the most recent calculation, output_factor parameter), 25 timestamps.


[2026-11-25 16:26:37,001] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 1 element(s) of element type ObjectTypes.WBLZ_ThermalBalance.


[2026-11-25 16:26:37,003] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties from the result file.


[2026-11-25 16:26:37,003] INFO in sir3stoolkit.mantle.dataframes: [results] 25 timestamp(s), 1 element(s), 2 properties resolved (0.0 s).


[2026-11-25 16:26:37,005] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=True)


[2026-11-25 16:26:37,009] INFO in sir3stoolkit.mantle.dataframes: [bulk results] 2 properties read from M-1-0-1.1.MX1 (0.0 s).


[2026-11-25 16:26:37,010] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:26:37,010] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:26:37,012] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 1 element(s) looked up (0.0 s).


[2026-11-25 16:26:37,012] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (25, 2) (0.0 s in total)


In [43]:
df_results.head()

tk                                      5086080348778933807          
name                                           WärmeblnzGes          
end_nodes                      No end nodes on element type          
property                                                WVB     WVERL
interior points                                          -1        -1
timestamp                                                            
2026-01-15 00:00:00.000 +01:00                     2.695431  0.263402
2026-01-15 01:00:00.000 +01:00                     2.488828  0.263093
2026-01-15 02:00:00.000 +01:00                     2.489110  0.263097
2026-01-15 03:00:00.000 +01:00                     2.489101  0.263096
2026-01-15 04:00:00.000 +01:00                     2.766253  0.263513

In [44]:
df_results.index

Index(['2026-01-15 00:00:00.000 +01:00', '2026-01-15 01:00:00.000 +01:00',
       '2026-01-15 02:00:00.000 +01:00', '2026-01-15 03:00:00.000 +01:00',
       '2026-01-15 04:00:00.000 +01:00', '2026-01-15 05:00:00.000 +01:00',
       '2026-01-15 06:00:00.000 +01:00', '2026-01-15 07:00:00.000 +01:00',
       '2026-01-15 08:00:00.000 +01:00', '2026-01-15 09:00:00.000 +01:00',
       '2026-01-15 10:00:00.000 +01:00', '2026-01-15 11:00:00.000 +01:00',
       '2026-01-15 12:00:00.000 +01:00', '2026-01-15 13:00:00.000 +01:00',
       '2026-01-15 14:00:00.000 +01:00', '2026-01-15 15:00:00.000 +01:00',
       '2026-01-15 16:00:00.000 +01:00', '2026-01-15 17:00:00.000 +01:00',
       '2026-01-15 18:00:00.000 +01:00', '2026-01-15 19:00:00.000 +01:00',
       '2026-01-15 20:00:00.000 +01:00', '2026-01-15 21:00:00.000 +01:00',
       '2026-01-15 22:00:00.000 +01:00', '2026-01-15 23:00:00.000 +01:00',
       '2026-01-16 00:00:00.000 +01:00'],
      dtype='object', name='timestamp')

# Limitations

If a single pipe is invalid (e.g. its length is 0 or negative), it can make an interior-point property unreadable for all pipes with bulk=True: the property is left out with a warning. bulk=False still returns it via the API.